In [1]:
!git lfs install
!git clone https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507 ./localqwen

Git LFS initialized.
Cloning into './localqwen'...
remote: Enumerating objects: 25, done.
remote: Total 25 (delta 0), reused 0 (delta 0), pack-reused 25 (from 1)
Receiving objects: 100% (25/25), 1.73 MiB | 5.41 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Filtering content: 100% (4/4), 7.50 GiB | 12.08 MiB/s, done.


In [2]:
!gdown 1X7MjI27W5mB5dxAeJPaOkFgnG1zXKEnW
!unzip ./ml-olympiad-red-task.zip

Downloading...
From (original): https://drive.google.com/uc?id=1X7MjI27W5mB5dxAeJPaOkFgnG1zXKEnW
From (redirected): https://drive.google.com/uc?id=1X7MjI27W5mB5dxAeJPaOkFgnG1zXKEnW&confirm=t&uuid=4a01edbd-e386-4fec-95bf-db108e5b53b6
To: /content/ml-olympiad-red-task.zip
100% 128M/128M [00:01<00:00, 101MB/s]
Archive:  ./ml-olympiad-red-task.zip
   creating: ml-olympiad-red-task/
  inflating: __MACOSX/._ml-olympiad-red-task  
   creating: ml-olympiad-red-task/metrics/
  inflating: __MACOSX/ml-olympiad-red-task/._metrics  
  inflating: ml-olympiad-red-task/УСЛОВИЯ.pdf  
  inflating: __MACOSX/ml-olympiad-red-task/._УСЛОВИЯ.pdf  
  inflating: ml-olympiad-red-task/.DS_Store  
  inflating: __MACOSX/ml-olympiad-red-task/._.DS_Store  
  inflating: ml-olympiad-red-task/УСЛОВИЯ.docx  
  inflating: __MACOSX/ml-olympiad-red-task/._УСЛОВИЯ.docx  
   creating: ml-olympiad-red-task/data/
  inflating: __MACOSX/ml-olympiad-red-task/._data  
  inflating: ml-olympiad-red-task/УСЛОВИЯ.md  
  inflating: __M

In [3]:
!pip install -q peft trl datasets bitsandbytes scikit-learn accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.8 MB/s eta 0:00:00


In [4]:
import os
import random
import numpy as np
import torch
import pickle
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from tqdm import tqdm

seed = 42

random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Задача 1

In [5]:
model_name = "./localqwen"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

lora_settings = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

train_model = get_peft_model(base_model, lora_settings)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [6]:
raw_data = load_dataset("json", data_files="ml-olympiad-red-task/data/kid_adult.jsonl", split="train")

def format(data):
    conversation = [
        {"role": "user", "content": data["prompt"]},
        {"role": "assistant", "content": data["kid"]}
    ]
    formatted_text = tokenizer.apply_chat_template(conversation, tokenize=False)
    return {"formatted_text": formatted_text}

train_data = raw_data.map(
    format,
    remove_columns=raw_data.column_names
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [7]:
train_args = SFTConfig(
    output_dir="./sft_train",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2.7e-4,
    max_steps=40,
    seed=42,
    dataset_text_field="formatted_text"
)

sft_trainer = SFTTrainer(
    model=train_model,
    train_dataset=train_data,
    args=train_args,
    processing_class=tokenizer
)

Adding EOS to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [8]:
sft_trainer.train()

train_model.save_pretrained("./sft_model")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss


In [12]:
test_data = load_dataset("json", data_files="ml-olympiad-red-task/data/public_test_style.jsonl", split="train")

train_model.eval()
generated_responses = []

with torch.no_grad():
    for row in tqdm(test_data):
        prompt_messages = [{"role": "user", "content": row['prompt']}]

        prompt_text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

        outputs = train_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

        answer_only = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        generated_responses.append(answer_only)

  0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 2/2 [01:14<00:00, 37.03s/it]


In [13]:
with open("ml-olympiad-red-task/metrics/style_clf.pkl", "rb") as f:
    style_evaluator = pickle.load(f)

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking c

In [14]:
import scipy.sparse as sp
features = []

for vec in style_evaluator['vecs']:
    transformed_data = vec.transform(generated_responses)
    features.append(transformed_data)

if len(features) > 1:
    vectorized_answers = sp.hstack(features)
else:
    vectorized_answers = features[0]

simple_prob = style_evaluator['clf'].predict_proba(vectorized_answers)[:, 1]

mean_p = np.mean(simple_prob)

print(f"SFT P_simple: {mean_p}")

SFT P_simple: 0.23727120541906832


# Задача 2

In [15]:
from trl import DPOConfig, DPOTrainer

In [16]:
import gc

del sft_trainer
gc.collect()
torch.cuda.empty_cache()

In [17]:
raw_data_dpo = load_dataset("json", data_files="ml-olympiad-red-task/data/kid_adult.jsonl", split="train")

def dpo_format(data):
    prompt_msg = [{"role": "user", "content": data["prompt"]}]
    prompt_text = tokenizer.apply_chat_template(prompt_msg, tokenize=False, add_generation_prompt=True)

    return {
        "prompt": prompt_text,
        "chosen": data["kid"] + tokenizer.eos_token,
        "rejected": data["adult"] + tokenizer.eos_token
    }


dpo_data = raw_data_dpo.map(dpo_format, remove_columns=raw_data_dpo.column_names)

Map:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [18]:
dpo_args = DPOConfig(
    output_dir="./dpo_train",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_steps=15,
    beta=0.1,
    seed=42
)

dpo_trainer = DPOTrainer(
    model=train_model,
    ref_model=None,
    train_dataset=dpo_data,
    processing_class=tokenizer,
    args=dpo_args
)

Adding EOS to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [19]:
dpo_trainer.train()

train_model.save_pretrained("./dpo_model")

Step,Training Loss


In [20]:
test_data = load_dataset("json", data_files="ml-olympiad-red-task/data/public_test_style.jsonl", split="train")

train_model.eval()
generated_responses = []

with torch.no_grad():
    for row in tqdm(test_data):
        prompt_messages = [{"role": "user", "content": row['prompt']}]

        prompt_text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

        outputs = train_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

        answer_only = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        generated_responses.append(answer_only)

  0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 2/2 [00:39<00:00, 19.79s/it]


In [21]:
features = []

for vec in style_evaluator['vecs']:
    transformed_data = vec.transform(generated_responses)
    features.append(transformed_data)

if len(features) > 1:
    vectorized_answers = sp.hstack(features)
else:
    vectorized_answers = features[0]

simple_prob = style_evaluator['clf'].predict_proba(vectorized_answers)[:, 1]

mean_p = np.mean(simple_prob)

print(f"DPO P_simple: {mean_p}")

DPO P_simple: 0.1490587759729246


# Я пропустил задачу 3, тк не разобрался

# Задача 4

In [22]:
del dpo_trainer
gc.collect()
torch.cuda.empty_cache()

In [23]:
raw_gb_data = load_dataset("json", data_files="ml-olympiad-red-task/data/good_bad.jsonl", split="train")

def gb_format(data):
    prompt_msg = [{"role": "user", "content": data["instruction"]}]
    prompt_text = tokenizer.apply_chat_template(prompt_msg, tokenize=False, add_generation_prompt=True)

    return {
        "prompt": prompt_text,
        "chosen": data["chosen"] + tokenizer.eos_token,
        "rejected": data["rejected"] + tokenizer.eos_token
    }

gb_data = raw_gb_data.map(gb_format, remove_columns=raw_gb_data.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2226 [00:00<?, ? examples/s]

In [24]:
gb_args = DPOConfig(
    output_dir="./gb_train",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=6e-5,
    max_steps=15,
    beta=0.1,
    seed=42
)

gb_trainer = DPOTrainer(
    model=train_model,
    ref_model=None,
    train_dataset=gb_data,
    processing_class=tokenizer,
    args=gb_args
)

Adding EOS to train dataset:   0%|          | 0/2226 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2226 [00:00<?, ? examples/s]

In [25]:
gb_trainer.train()

train_model.save_pretrained("./gb_model")

Step,Training Loss


In [26]:
import torch.nn.functional as F

quality_data = load_dataset("json", data_files="ml-olympiad-red-task/data/public_test_quality.jsonl", split="train")

def mean_logprob(prompt, answer, model, tokenizer):
    msg = [{"role": "user", "content": prompt}]
    prompt_formatted = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)

    prompt_tokens = tokenizer(prompt_formatted, return_tensors="pt")["input_ids"]
    prompt_len = prompt_tokens.shape[1]

    full_text = prompt_formatted + answer + tokenizer.eos_token
    inputs = tokenizer(full_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    shift_logits = logits[0, prompt_len-1:-1, :]
    shift_labels = inputs["input_ids"][0, prompt_len:]

    log_probs = F.log_softmax(shift_logits, dim=-1)

    res_log_probs = log_probs.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)

    return res_log_probs.mean().item()

Generating train split: 0 examples [00:00, ? examples/s]

In [27]:
train_model.eval()

correct_predictions = 0
total_pairs = len(quality_data)

for row in tqdm(quality_data):
    prompt = row["prompt"]
    good_ans = row["chosen"]
    bad_ans = row["rejected"]

    score_good = mean_logprob(prompt, good_ans, train_model, tokenizer)
    score_bad = mean_logprob(prompt, bad_ans, train_model, tokenizer)

    if score_good > score_bad:
        correct_predictions += 1

accuracy = correct_predictions / total_pairs

print(f"Implicit-preference accuracy DPO good_bad {accuracy}")

100%|██████████| 2/2 [00:03<00:00,  1.81s/it]

Implicit-preference accuracy DPO good_bad 0.0
